# Key Learnings — HDB Resale Price Analysis

Compiled from four analysis notebooks:
- **P1** — Correlation Analysis & Baseline Model
- **P2** — Exploratory Data Analysis on `revised_train.csv`
- **P3** — Model Building (Optuna-tuned, advanced features)
- **P4** — Baseline Model (fixed hyperparameters, simple features)

**Dataset:** 150,634 HDB resale transactions (2012–2021), 77 raw columns.

---
## 1. Data Cleaning & Feature Engineering (P1)

### 1.1 Spatial Feature Recomputation
- The original hawker/mall proximity columns had **inconsistencies**. They were recomputed from scratch using a **BallTree** (haversine metric) with coordinates from `HawkerCentresGEOJSON.geojson` and `shopping_mall_coordinates.csv`.
- Counts within 500 m, 1 km, and 2 km were recalculated, and `Mall_Nearest_Distance` was recomputed as the haversine distance to the closest mall.
- **Lesson:** Always validate spatial features against source data — pre-supplied proximity columns may be stale or computed inconsistently.

### 1.2 Derived Features
- **`lease_remaining_year`** = `lease_commence_date + 99 - Tranc_Year` — captures the remaining lease at the time of sale, which is a strong predictor (Pearson r ≈ 0.36 with `resale_price`).

### 1.3 Column Cleanup
- **23 of 77 columns dropped:** identifiers (`id`), redundant size measures (`floor_area_sqft`, `upper`, `mid`, `lower`, `hdb_age`), text fields (`address`, `storey_range`), amenity coordinates (lat/lon for MRT, bus stop, schools), and lookup keys (`year_completed`, `mrt_name`).
- Y/N categorical columns (`residential`, `commercial`, etc.) were encoded to 0/1. `residential` was then dropped because it had only one unique value (all "Y").
- **Final cleaned dataset:** 150,634 × 54 → saved as `revised_train.csv`.
- **Lesson:** Aggressively remove columns that are identifiers, single-valued, or redundant with existing distance features. Keep the dataset lean.

---
## 2. Correlation Insights (P1)

| Rank | Feature | Pearson r |
|------|---------|----------|
| 1 | `floor_area_sqm` | 0.654 |
| 2 | `max_floor_lvl` | 0.496 |
| 3 | `3room_sold` | 0.410 |
| 4 | `lease_remaining_year` | 0.362 |
| 5 | `5room_sold` | 0.359 |
| 6 | `mid_storey` | 0.353 |
| 7 | `exec_sold` | 0.338 |
| 8 | `Latitude` | 0.216 |
| 9 | `Hawker_Within_2km` | 0.201 |

- **`floor_area_sqm`** dominates — larger flats cost more. This remains the #1 feature across all models.
- **`lease_remaining_year`** is the strongest non-size predictor, reflecting buyers' sensitivity to depreciating leasehold value.
- **Location proxies** (Latitude, hawker/mall counts) show moderate correlations, confirming that town/neighbourhood matters.
- **Lesson:** Simple linear correlations already highlight the most important features. Use them as a quick sanity check before modelling.

---
## 3. EDA Insights (P2)

### 3.1 Price Distribution
- `resale_price_per_sqm` is **right-skewed**; a log transform makes it approximately normal — useful if you need normality assumptions (e.g. linear regression residuals).

### 3.2 Temporal Trends
- Prices dipped ~2014–2017 (cooling measures), then recovered strongly through 2021.
- All towns follow a **similar directional trend**, but at different price levels — Central Area and Bukit Timah sit consistently at the top; Woodlands and Jurong West at the bottom.

### 3.3 `time_index` Feature
- Normalised average price per sqm by the **Woodlands-2012 baseline** (= 1.0). This collapses temporal inflation and cross-town premiums into a single scalar per town-year cell.
- `time_index` became one of the **top-5 most important features** in every tree model.
- **Lesson:** Creating a relative price index anchored to a reference town-year is a powerful way to encode macro-level market conditions.

### 3.4 Amenity Proximity
- More malls/hawker centres nearby → higher price per sqm. The effect is strongest within 500 m.
- Distance-band features (`Mall_500m_1km`, `Mall_1km_2km`) isolate incremental amenity counts between rings.
- ~29,000 rows have zero malls within 1 km. No negative values in band columns (data quality check passed).

### 3.5 Transaction Volume
- The heatmap of records per town × year shows that **Woodlands, Jurong West, Sengkang, and Tampines** have the most transactions, while **Bukit Timah** (369 rows) and **Marine Parade** (959 rows) are sparse — averages for those towns may be noisy.

---
## 4. Baseline Model Results (P1)

Using all 45 numeric features, an 80/20 train/val split, and no hyperparameter tuning:

| Model | Validation RMSE | R² |
|-------|----------------|----|
| Ridge Regression | 60,728 | 0.819 |
| Random Forest | **26,372** | **0.966** |
| Gradient Boosting | 28,282 | 0.961 |

- **Lesson:** Tree-based models immediately outperform linear models by a wide margin on this dataset. The non-linear interactions between floor area, storey, lease, and location are critical.

---
## 5. Advanced Modelling Results (P3 vs P4)

### 5.1 P4 — Baseline (fixed hyperparams, simple features)

| Model | 5-Fold OOF RMSE |
|-------|----------------|
| LightGBM | 22,578 |
| XGBoost | 22,443 |
| CatBoost | **22,282** |
| Blend (weighted) | 22,234 |
| **Blend (optimised)** | **22,172** |

Optimised blend weights: LightGBM 0.03, XGBoost 0.36, **CatBoost 0.61**.

### 5.2 P3 — Optuna-tuned + advanced features

| Model | 5-Fold OOF RMSE |
|-------|----------------|
| CatBoost | 22,420 |
| LightGBM | 22,566 |
| XGBoost | 22,920 |
| Random Forest | 24,725 |
| Ridge | 46,016 |
| ElasticNet | 103,398 |
| Blend (weighted top 3) | 22,387 |
| **Blend (optimised all)** | **22,314** |
| Stacking (Ridge meta) | 119,883 |
| Stacking (trees only) | 97,286 |

### 5.3 Key Comparisons

- **P4 baseline (22,172) beat P3 advanced (22,314)** despite P3 having Optuna tuning, 20 extra engineered features, and target encoding.
- **Lesson:** More features and more tuning do not guarantee better results. The additional features in P3 (squared terms, interactions, target encoding) likely introduced noise or overfitting.
- **CatBoost is the strongest individual model** in both notebooks, likely due to its native handling of categorical features.
- **Stacking failed badly** — the Ridge meta-learner was unstable across folds, producing RMSE > 97k. This is likely because OOF predictions from correlated tree models do not provide enough diversity for a linear stacker.
- **Scipy-optimised blending** consistently beats equal/inverse-RMSE weighting. It shifts weight heavily toward CatBoost (0.61 in P4).

---
## 6. Feature Importance (Tree Models)

### P4 (Baseline) — LightGBM top features:
1. `floor_area_sqm` — 58,387
2. `time_index` — 47,607
3. `lease_remaining_year` — 41,705
4. `storey_ratio` — 40,357
5. `Tranc_Year` — 36,096
6. `mrt_nearest_distance` — 30,500
7. `bus_stop_nearest_distance` — 29,430

### P3 (Advanced) — LightGBM top features:
1. `lease_x_area` — 38,285
2. `time_x_area` — 38,171
3. `lease_x_storey` — 31,657
4. `storey_x_area` — 29,738
5. `time_index` — 27,434

- **Lesson:** In P3, interaction features (lease×area, time×area) rose to the top, but they **redistributed** importance rather than adding new signal — the underlying base features (`floor_area_sqm`, `lease_remaining_year`, `time_index`) are the true drivers.
- **`time_index`** is consistently top-5 — validating the EDA decision to engineer this feature.
- Distance to amenities (MRT, bus stop, schools, malls, hawkers) collectively form a strong signal block.

---
## 7. Overall Lessons & Recommendations

### What worked
- **BallTree recomputation** of spatial features from source coordinates ensured data consistency.
- **`time_index`** — a simple normalised price index — became one of the most important features.
- **CatBoost** consistently won among individual models, benefiting from native categorical handling.
- **Scipy-optimised blending** of 3 tree models reliably reduced RMSE by ~100–200 over the best single model.
- **5-Fold OOF evaluation** provided stable, unbiased estimates vs. a single train/val split.

### What did not work
- **Advanced feature engineering (P3):** Squared terms, interaction features, target encoding, and log transforms did not improve over the simpler P4 baseline. They likely added noise.
- **Stacking with Ridge meta-learner:** Unstable and drastically worse than simple blending. The correlated base predictions do not provide enough diversity.
- **Linear models (Ridge, ElasticNet):** RMSE 46k–103k — far too weak for this non-linear problem.
- **Optuna tuning (P3 vs P4):** 50 trials per model did not beat fixed "sensible default" hyperparameters.

### Gap to target
- **Target RMSE: 21,200.** Best achieved: **22,172** (P4 optimised blend). Gap: **~972.**
- To close this gap, consider:
  - **More data:** external features (e.g. GDP, interest rates, MRT opening dates).
  - **Spatial features:** latitude/longitude clustering, or neighbourhood-level embeddings.
  - **Time-series features:** rolling averages, momentum indicators for each town.
  - **Better stacking:** use a non-linear meta-learner (e.g. LightGBM stacker with careful out-of-fold leakage prevention).
  - **Hyperparameter search on a larger budget** with proper nested cross-validation.

---
## 8. Data Pipeline Summary

```
train.csv (77 cols)
  |
  +-- P1: BallTree recompute -> encode Y/N -> drop 23 cols -> lease_remaining_year
  |       -> revised_train.csv (54 cols)
  |
  +-- P2: EDA -> time_index -> distance bands -> drop 3 cols / add 3 cols
  |       -> revised2_train.csv (54 cols) -> windsurf_train.csv
  |
  +-- P4: Simple FE -> label encode -> LGB/XGB/CatBoost -> blend
  |       -> Best RMSE: 22,172
  |
  +-- P3: Advanced FE -> target encode -> Optuna -> 6 models -> blend/stack
          -> Best RMSE: 22,314
```